
# Análisis de Componentes Principales (PCA)

Este notebook muestra como calcular las componentes principales del archivo `europe.csv` e interpretar la **primera componente principal (PC1)** de forma gráfica y teórica.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True



## 1. Carga de datos

El notebook busca `europe.csv` en una carpeta `data/`. 


In [ ]:
csv_path = Path("data/europe.csv")

print(f"Archivo cargado desde: {csv_path}")

df = pd.read_csv(csv_path)
df.head()

In [ ]:

df.describe().T.round(2)



## 2. Preprocesamiento: selección de variables y estandarización

PCA es sensible a la escala de las variables. Por ejemplo, `Area` puede tomar valores de cientos de miles, mientras que `Inflation` o `Pop.growth` son porcentajes. Si no se estandarizan los datos, las variables con mayor magnitud numérica dominarían artificialmente las componentes.

Por eso se aplica `StandardScaler`, que transforma cada variable para que tenga media 0 y desvío estándar 1.


In [ ]:

features = [col for col in df.columns if col != "Country"]
X = df[features]
countries = df["Country"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=features, index=countries)
X_scaled_df.head().round(2)



## 3. Cálculo de PCA con `scikit-learn`

Se calculan todas las componentes principales.


In [ ]:

pca = PCA()
scores = pca.fit_transform(X_scaled)

component_names = [f"PC{i+1}" for i in range(len(features))]

# Coeficientes de los vectores propios: cuánto pesa cada variable en cada PC.
coefficients = pd.DataFrame(
    pca.components_.T,
    columns=component_names,
    index=features,
)


scores_df = pd.DataFrame(scores, columns=component_names)
scores_df.insert(0, "Country", countries)

explained_variance = pd.DataFrame({
    "Componente": component_names,
    "Varianza explicada": pca.explained_variance_ratio_,
    "Varianza acumulada": np.cumsum(pca.explained_variance_ratio_),
})

explained_variance.round(4)


In [ ]:

pc1_pct = explained_variance.loc[0, "Varianza explicada"] * 100
pc2_pct = explained_variance.loc[1, "Varianza explicada"] * 100

print(f"PC1 explica {pc1_pct:.2f}% de la varianza total.")
print(f"PC2 explica {pc2_pct:.2f}% de la varianza total.")
print(f"PC1 + PC2 explican {(pc1_pct + pc2_pct):.2f}% de la varianza total.")



### Varianza explicada

La primera componente principal resume una parte importante de la información del dataset. En este caso, **PC1 explica aproximadamente el 46% de la varianza total** de las variables estandarizadas.

Esto no significa que PC1 explique “todo” el comportamiento de los países, sino que captura la dirección de mayor variabilidad conjunta entre las variables económicas, sociales y geográficas.


In [ ]:

plt.figure(figsize=(9, 5))
plt.plot(
    explained_variance["Componente"],
    explained_variance["Varianza explicada"] * 100,
    marker="o",
)
plt.title("Varianza explicada por componente principal")
plt.xlabel("Componente principal")
plt.ylabel("Varianza explicada (%)")
plt.show()


In [ ]:

plt.figure(figsize=(9, 5))
plt.plot(
    explained_variance["Componente"],
    explained_variance["Varianza acumulada"] * 100,
    marker="o",
)
plt.title("Varianza explicada acumulada")
plt.xlabel("Cantidad de componentes")
plt.ylabel("Varianza acumulada (%)")
plt.ylim(0, 105)
plt.show()



## 4. Interpretación de PC1

Para interpretar PC1 conviene distinguir dos conceptos relacionados pero no idénticos:

1. **Coeficientes de la componente**: son los pesos del vector propio de PC1. Indican cómo se combina cada variable estandarizada para construir la componente.
2. **Correlaciones variable-PC1**: miden la correlación entre cada variable original estandarizada y los scores de PC1.

La interpretación cualitativa suele mirar signos y magnitudes. Variables con signo positivo se mueven en la dirección positiva de PC1, mientras que variables con signo negativo se mueven en dirección opuesta.


In [ ]:

pc1_table = pd.DataFrame({
    "Coeficiente en PC1": coefficients["PC1"],
})

# Correlaciones entre las variables estandarizadas y los scores de PC1.
pc1_table["Correlación con PC1"] = [
    np.corrcoef(X_scaled_df[var], scores_df["PC1"])[0, 1]
    for var in features
]

pc1_table["Magnitud abs. coeficiente"] = pc1_table["Coeficiente en PC1"].abs()
pc1_table = pc1_table.sort_values("Magnitud abs. coeficiente", ascending=False)
pc1_table.round(4)


In [ ]:

pc1_coefficients = coefficients["PC1"].sort_values()

plt.figure(figsize=(9, 5))
plt.barh(pc1_coefficients.index, pc1_coefficients.values)
plt.axvline(0, linewidth=1)
plt.title("Coeficientes de las variables en PC1")
plt.xlabel("Coeficiente")
plt.ylabel("Variable")
plt.show()



### Interpretación teórica de PC1

PC1 tiene pesos positivos fuertes en:

- `GDP`: mayor riqueza económica.
- `Life.expect`: mayor expectativa de vida, asociada a mejores condiciones de bienestar y salud.
- `Pop.growth`: crecimiento poblacional, que en este contexto puede asociarse a dinámicas demográficas más favorables o a atracción migratoria.

Y pesos negativos relevantes en:

- `Inflation`: mayor inflación.
- `Unemployment`: mayor desempleo.
- `Military`: menor contribución relativa, pero con signo negativo en PC1.

Por lo tanto, **PC1 puede interpretarse como un eje de desarrollo y estabilidad socioeconómica**.

- Valores altos de PC1: países con mayor GDP, mayor expectativa de vida y mejores indicadores generales.
- Valores bajos de PC1: países con mayor inflación, mayor desempleo y menor desarrollo relativo dentro del conjunto analizado.




## 5. Países extremos según PC1

Para validar la interpretación, observamos qué países tienen los scores más altos y más bajos en PC1.


In [ ]:

pc_scores_ordered = scores_df[["Country", "PC1", "PC2"]].sort_values("PC1", ascending=False).round(4)

print("Países con PC1 más alta:")
display(pc_scores_ordered.head(8))

print("Países con PC1 más baja:")
display(pc_scores_ordered.tail(8).sort_values("PC1"))



Los países con PC1 más alta aparecen asociados al polo de mayor desarrollo socioeconómico relativo. En este dataset suelen ubicarse allí países como **Luxemburgo**, **Suiza**, **Noruega**, **Países Bajos** o **Irlanda**.

En el extremo negativo aparecen países como **Ucrania**, **Bulgaria**, **Estonia**, **Letonia** o **Lituania**, que en estos datos presentan peores valores relativos en las variables que más empujan PC1 hacia el lado negativo.



## 6. Visualización gráfica de PC1 y PC2

Primero se grafican los países proyectados sobre PC1 y PC2. Luego se construye un biplot que agrega flechas para las variables originales.


In [ ]:

plt.figure(figsize=(12, 8))
plt.scatter(scores_df["PC1"], scores_df["PC2"], s=70)

for _, row in scores_df.iterrows():
    plt.annotate(
        row["Country"],
        (row["PC1"], row["PC2"]),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=9,
    )

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.title("Países proyectados en el plano PC1-PC2")
plt.xlabel(f"PC1 ({pc1_pct:.2f}% de varianza) — desarrollo / estabilidad socioeconómica")
plt.ylabel(f"PC2 ({pc2_pct:.2f}% de varianza)")
plt.show()


In [ ]:

def plot_biplot(scores_df, coefficients, features, pc1_pct, pc2_pct):
    # Biplot simple para PC1 y PC2.
    x = scores_df["PC1"].to_numpy()
    y = scores_df["PC2"].to_numpy()

    # Escalado de scores para que entren visualmente con las flechas.
    x_scaled = x / np.max(np.abs(x))
    y_scaled = y / np.max(np.abs(y))

    plt.figure(figsize=(13, 9))
    plt.scatter(x_scaled, y_scaled, s=70, alpha=0.8)

    for i, country in enumerate(scores_df["Country"]):
        plt.annotate(country, (x_scaled[i], y_scaled[i]), xytext=(5, 4), textcoords="offset points", fontsize=9)

    # Flechas de variables: se usan los coeficientes de PC1 y PC2.
    for var in features:
        x_arrow = coefficients.loc[var, "PC1"]
        y_arrow = coefficients.loc[var, "PC2"]
        plt.arrow(0, 0, x_arrow, y_arrow, head_width=0.03, length_includes_head=True)
        plt.text(x_arrow * 1.12, y_arrow * 1.12, var, ha="center", va="center", fontsize=11, fontweight="bold")

    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.title("Biplot PCA: países y variables")
    plt.xlabel(f"PC1 ({pc1_pct:.2f}%) — desarrollo / estabilidad socioeconómica")
    plt.ylabel(f"PC2 ({pc2_pct:.2f}%)")
    plt.xlim(-1.2, 1.2)
    plt.ylim(-1.2, 1.2)
    plt.show()

plot_biplot(scores_df, coefficients, features, pc1_pct, pc2_pct)



### Lectura del biplot

En el biplot, las flechas de `GDP`, `Life.expect` y `Pop.growth` apuntan hacia el lado positivo de PC1. Esto confirma gráficamente que el eje horizontal puede leerse como un eje de mayor desarrollo y estabilidad socioeconómica.

En sentido contrario se ubican `Inflation` y `Unemployment`, que empujan hacia el lado negativo de PC1. Por eso, los países ubicados a la izquierda del gráfico tienden a presentar peores condiciones relativas en esos indicadores.

La PC2 explica menos varianza que PC1 y su interpretación es menos clara. En estos datos, PC2 tiene contribuciones importantes de `Military` y `Unemployment`, por lo que no conviene sobreinterpretarla como “tamaño geográfico” o “poder geopolítico”. El análisis central de la consigna debe enfocarse en PC1.



## 7. Conclusión final

Luego de estandarizar las variables y aplicar PCA con `scikit-learn`, se observa que la primera componente principal explica aproximadamente el **46% de la varianza total** de las variables estandarizadas.

Al analizar los coeficientes de PC1, las variables con mayor peso positivo son `GDP`, `Life.expect` y `Pop.growth`, mientras que `Inflation` y `Unemployment` aparecen con pesos negativos. Por lo tanto, con la orientación elegida, **PC1 puede interpretarse como un eje de desarrollo y estabilidad socioeconómica**.

Gráficamente, esta interpretación se observa en el plano PC1-PC2: países como **Luxemburgo**, **Suiza** y **Noruega** aparecen hacia el lado positivo de PC1, mientras que países como **Ucrania**, **Bulgaria**, **Estonia** o **Letonia** aparecen hacia el lado negativo.

En síntesis, PC1 resume una oposición entre países con mejores indicadores económicos y sociales frente a países con mayores señales de inestabilidad macroeconómica o laboral.
